# Notebook 2 — Tokenizers Deep Dive

### AI Builders Bootcamp

In Notebook 1 we treated the tokenizer as a black box: `AutoTokenizer.from_pretrained(...)` just
*worked*. Today we open that box.

By the end you will be able to:

- Explain **why** tokenization exists at all, and the character → word → subword progression
- Implement **character** and **word** tokenization from scratch, and see why both fail at scale
- Implement a **tiny Byte-Pair Encoding (BPE)** algorithm from scratch, merge by merge
- Explain **WordPiece**, **SentencePiece**, and **Unigram** and demonstrate each with real tokenizers
- Compare tokenizers across **8 model families** (GPT-2, Llama, Gemma, BERT, RoBERTa, T5, Mistral, Qwen)
- Explain every field returned by a tokenizer call (`input_ids`, `attention_mask`, `token_type_ids`, ...)
- Train your **own tokenizer from scratch** on a tiny dataset using the `tokenizers` library

> Some tokenizers here are **gated** on the Hugging Face Hub (Llama, Gemma, Mistral). You'll need a
> free HF account, to accept each model's license on its Hub page, and to log in with
> `notebook_login()` below. If you skip this, every other section still works — you'll just see a
> note to skip those specific comparison rows.

Let's take tokenization apart. 🔬


## 1. Setup

In [ ]:
!pip install -q "transformers>=4.46" tokenizers datasets huggingface_hub

import transformers
print("transformers:", transformers.__version__)


transformers: 5.13.1


### 👉 TODO — log in to the Hugging Face Hub
Some tokenizers we'll compare (Llama 3, Gemma, Mistral) are **gated**: you must accept their
license on huggingface.co before you can download them.

1. Create a free account at huggingface.co if you don't have one.
2. Visit each model page and click "Agree and access repository":
   `meta-llama/Llama-3.2-1B`, `google/gemma-2-2b`, `mistralai/Mistral-7B-v0.3`.
3. Run the cell below and paste a token from huggingface.co/settings/tokens.

If you'd rather skip this, that's fine — set `HAVE_HF_ACCESS = False` and we'll skip the gated
rows in the comparison table later.


### 👉 TODO — log in to the Hugging Face Hub
Some tokenizers we'll compare (Llama 3, Gemma, Mistral) are **gated**: you must accept their
license on huggingface.co before you can download them.

1. Create a free account at huggingface.co if you don't have one.
2. Visit each model page and click "Agree and access repository":
   `meta-llama/Llama-3.2-1B`, `google/gemma-2-2b`, `mistralai/Mistral-7B-v0.3`.
3. Run the cell below and paste a token from huggingface.co/settings/tokens.

If you'd rather skip this, that's fine — set `HAVE_HF_ACCESS = False` and we'll skip the gated
rows in the comparison table later.


In [ ]:
HAVE_HF_ACCESS = True  # flip to False to skip gated models

if HAVE_HF_ACCESS:
    from huggingface_hub import notebook_login
    notebook_login()


## 2. Why tokenization exists

Neural networks only understand **numbers**. Text is made of characters, but a model needs a
finite, fixed-size vocabulary of "chunks" it can look up in an embedding table. Tokenization is
the process of deciding **what those chunks are**.

```
Characters
    │   (too fine-grained: sequences explode, no semantic meaning per unit)
    ▼
 Words
    │   (too coarse: vocabulary explodes, can't handle new/rare words)
    ▼
Subwords
    │   (sweet spot: reusable pieces, manageable vocabulary, handles anything)
    ▼
 Tokens  ──►  integers (input_ids)  ──►  embedding lookup  ──►  model
```

Let's feel *why* the middle ground (subwords) won, by implementing the two extremes ourselves.


## 3. Character tokenization

The simplest possible scheme: every character is a token.


In [ ]:
def char_tokenize(text):
    return list(text)

text = "Tokenization is fun!"
char_tokens = char_tokenize(text)
print(char_tokens)
print("Number of tokens:", len(char_tokens))

char_vocab = sorted(set(text))
print("\nVocabulary:", char_vocab)
print("Vocabulary size:", len(char_vocab))


['T', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n', ' ', 'i', 's', ' ', 'f', 'u', 'n', '!']
Number of tokens: 20

Vocabulary: [' ', '!', 'T', 'a', 'e', 'f', 'i', 'k', 'n', 'o', 's', 't', 'u', 'z']
Vocabulary size: 14


### Disadvantages
- **Sequences get very long** — a 20-character word is 20 tokens instead of ~1-3. Transformers'
  compute cost grows quadratically with sequence length (self-attention), so this is expensive.
- **Individual characters carry almost no meaning.** The model has to learn to *compose* meaning
  from scratch out of `["c", "a", "t"]` — much harder than starting from a token that already
  means "cat".

### 🧩 Quiz
How many tokens would `"internationalization"` (20 characters) produce under character
tokenization? What about under a hypothetical scheme that treated the whole word as one token?


## 4. Word tokenization

The other extreme: split on whitespace/punctuation, one token per word.


In [ ]:
import re

def word_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)

text = "Tokenization is fun, isn't it?"
word_tokens = word_tokenize(text)
print(word_tokens)


['Tokenization', 'is', 'fun', ',', 'isn', "'", 't', 'it', '?']


### The out-of-vocabulary (OOV) problem

A word-level vocabulary is built from a fixed training corpus. What happens when we see a word
that **never appeared during training**?


In [ ]:
training_vocab = {"tokenization", "is", "fun", "isn't", "it", "?", ",", "the", "cat", "sat"}

def word_tokenize_with_vocab(text, vocab):
    tokens = word_tokenize(text)
    return [t if t.lower() in vocab else "[UNK]" for t in tokens]

test_text = "Tokenization is fun, but subwording is superior!"
print(word_tokenize_with_vocab(test_text, training_vocab))


['Tokenization', 'is', 'fun', ',', '[UNK]', '[UNK]', 'is', '[UNK]', '[UNK]']


Notice `"subwording"` and `"but"` and `"superior"` all collapse into `[UNK]` — the model loses
**all** information about them, even though `"subwording"` obviously contains `"sub"` + `"word"` +
`"ing"`, pieces the model might well have seen elsewhere. This is the **out-of-vocabulary (OOV)
problem**, and it's the single biggest reason pure word tokenization lost to subwords.

### 🧠 Discuss
A word-level vocabulary big enough to cover most of English still needs to be **huge** (hundreds
of thousands of entries, more once you count names, typos, and morphological variants). What
does a bigger vocabulary cost the model? (Hint: where does vocabulary size show up in the model's
parameter count?)

> Every vocabulary entry needs its own row in the embedding matrix (`vocab_size × hidden_size`
> parameters) — and often a matching row in the output projection. A 500k-word vocabulary at a
> modest `hidden_size=768` is already ~**384M parameters** just for embeddings, before a single
> transformer layer.


## 5. Subword tokenization — the middle ground

Subword tokenization breaks **rare** words into smaller, **reusable** pieces, while keeping
**common** words as single tokens. `"tokenization"` might become `["token", "ization"]`; `"the"`
stays `["the"]`.

This gives us:
- A **fixed, manageable vocabulary** (typically 30k–150k entries)
- **No OOV problem** — worst case, an unseen word falls back to individual bytes/characters
- **Shared substructure** — `"ization"` in `"tokenization"` and `"modernization"` reuses the same
  token, so the model doesn't have to relearn the suffix's meaning from scratch each time

Three dominant algorithms build subword vocabularies differently: **BPE**, **WordPiece**, and
**Unigram** (used by SentencePiece). Let's build the first one ourselves.


## 6. Byte-Pair Encoding (BPE), from scratch

BPE's idea, in one sentence: **start from characters, and repeatedly merge the most frequent
adjacent pair into a new token — a fixed number of times.**

```
"low", "lower", "lowest", "newer", "newest"   (each split into characters)
      │
      ▼  find most frequent adjacent pair → merge it → repeat
      │
new tokens: "e r" -> "er", then "er w" is never adjacent... "l o" -> "lo", "lo w" -> "low", ...
```

Let's implement this on a tiny toy corpus so every merge is visible.


In [ ]:
from collections import Counter, defaultdict

# Toy corpus: word -> frequency. Each word is represented as a tuple of characters + an
# end-of-word marker "</w>" so the model can tell "est" at a word boundary from "est" mid-word.
corpus = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
    "lowest": 4,
}

word_freqs = {tuple(word) + ("</w>",): freq for word, freq in corpus.items()}
for word, freq in word_freqs.items():
    print(word, "->", freq)


('l', 'o', 'w', '</w>') -> 5
('l', 'o', 'w', 'e', 'r', '</w>') -> 2
('n', 'e', 'w', 'e', 's', 't', '</w>') -> 6
('w', 'i', 'd', 'e', 's', 't', '</w>') -> 3
('l', 'o', 'w', 'e', 's', 't', '</w>') -> 4


In [ ]:
def get_pair_counts(word_freqs):
    """Count how often each adjacent symbol pair occurs across the corpus."""
    pairs = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += freq
    return pairs

def merge_pair(pair, word_freqs):
    """Replace every adjacent occurrence of `pair` with a single merged symbol."""
    new_word_freqs = {}
    bigram = pair[0] + pair[1]
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(bigram)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs


In [ ]:
num_merges = 10
merges = []
current = dict(word_freqs)

for step in range(num_merges):
    pair_counts = get_pair_counts(current)
    if not pair_counts:
        break
    best_pair = max(pair_counts, key=pair_counts.get)
    current = merge_pair(best_pair, current)
    merges.append(best_pair)
    print(f"Merge {step + 1}: {best_pair} (count={pair_counts[best_pair]})  ->  {best_pair[0]+best_pair[1]}")

print("\nFinal tokenized corpus:")
for word in current:
    print(" ", word)


Merge 1: ('e', 's') (count=13)  ->  es
Merge 2: ('es', 't') (count=13)  ->  est
Merge 3: ('est', '</w>') (count=13)  ->  est</w>
Merge 4: ('l', 'o') (count=11)  ->  lo
Merge 5: ('lo', 'w') (count=11)  ->  low
Merge 6: ('n', 'e') (count=6)  ->  ne
Merge 7: ('ne', 'w') (count=6)  ->  new
Merge 8: ('new', 'est</w>') (count=6)  ->  newest</w>
Merge 9: ('low', '</w>') (count=5)  ->  low</w>
Merge 10: ('low', 'est</w>') (count=4)  ->  lowest</w>

Final tokenized corpus:
  ('low</w>',)
  ('low', 'e', 'r', '</w>')
  ('newest</w>',)
  ('w', 'i', 'd', 'est</w>')
  ('lowest</w>',)




Final tokenized corpus:
  ('low</w>',)
  ('low', 'e', 'r', '</w>')
  ('newest</w>',)
  ('w', 'i', 'd', 'est</w>')
  ('lowest</w>',)


  Final tokenized corpus:
  ('low</w>',)
  ('lower</w>',)
  ('newest</w>',)
  ('widest</w>',)
  ('lowest</w>',)

In [ ]:
num_merges = 20
merges = []
current = dict(word_freqs)

for step in range(num_merges):
    pair_counts = get_pair_counts(current)
    if not pair_counts:
        break
    best_pair = max(pair_counts, key=pair_counts.get)
    current = merge_pair(best_pair, current)
    merges.append(best_pair)
    print(f"Merge {step + 1}: {best_pair} (count={pair_counts[best_pair]})  ->  {best_pair[0]+best_pair[1]}")

print("\nFinal tokenized corpus:")
for word in current:
    print(" ", word)


Merge 1: ('e', 's') (count=13)  ->  es
Merge 2: ('es', 't') (count=13)  ->  est
Merge 3: ('est', '</w>') (count=13)  ->  est</w>
Merge 4: ('l', 'o') (count=11)  ->  lo
Merge 5: ('lo', 'w') (count=11)  ->  low
Merge 6: ('n', 'e') (count=6)  ->  ne
Merge 7: ('ne', 'w') (count=6)  ->  new
Merge 8: ('new', 'est</w>') (count=6)  ->  newest</w>
Merge 9: ('low', '</w>') (count=5)  ->  low</w>
Merge 10: ('low', 'est</w>') (count=4)  ->  lowest</w>
Merge 11: ('w', 'i') (count=3)  ->  wi
Merge 12: ('wi', 'd') (count=3)  ->  wid
Merge 13: ('wid', 'est</w>') (count=3)  ->  widest</w>
Merge 14: ('low', 'e') (count=2)  ->  lowe
Merge 15: ('lowe', 'r') (count=2)  ->  lower
Merge 16: ('lower', '</w>') (count=2)  ->  lower</w>

Final tokenized corpus:
  ('low</w>',)
  ('lower</w>',)
  ('newest</w>',)
  ('widest</w>',)
  ('lowest</w>',)


### 👉 TODO
Increase `num_merges` to `20` and re-run. What happens to `"lower"`, `"lowest"`, and `"newest"` —
do they collapse into single tokens? At what merge step does that happen?

### ✅ Checkpoint
After 10 merges, `"low"` and `"lowest"` should share a common `low` (or similar) prefix token —
this is BPE's core payoff: reused substructure across related words.


In [ ]:
# Your experiment: change num_merges above and re-run, or copy the loop here with a new value.


### Now compare with Hugging Face's real BPE tokenizer (GPT-2)

GPT-2's tokenizer is BPE trained on a huge web corpus, with **byte-level** encoding (so it never
hits an unknown character — worst case it falls back to raw bytes). Notice the `Ġ` symbol below:
it's how GPT-2's byte-level BPE represents a leading space.


In [ ]:
from transformers import AutoTokenizer

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
tokens = gpt2_tok.tokenize("lower lowest newest widest")
print(tokens)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

['lower', 'Ġlowest', 'Ġnewest', 'Ġwidest']


### 🧠 Discuss
Our toy BPE and GPT-2's BPE use the exact same *algorithm* (frequency-based pair merging). What's
different is **scale**: GPT-2 trained on billions of words with ~50,000 merges, while ours did 10
merges on 5 words. Same idea, different amount of data — this is a pattern you'll see across all
of deep learning.


## 7. WordPiece

**WordPiece** (used by BERT) is close cousin to BPE, with one key difference in *how it picks
which pair to merge*: instead of raw frequency, it scores each candidate merge by

```
score(pair) = frequency(pair) / (frequency(first) * frequency(second))
```

This favors merging pairs where the **combination is much more common than its parts would
suggest** — it prefers merges that make statistically informative units, not just frequent ones.

We won't reimplement WordPiece's training from scratch (it needs a large corpus to be meaningful)
— instead, let's use BERT's real tokenizer and **recognize WordPiece's signature**: the `##`
continuation prefix.


In [ ]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

for word in ["tokenization", "unhappiness", "supercalifragilisticexpialidocious"]:
    print(f"{word:>35} -> {bert_tok.tokenize(word)}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

                       tokenization -> ['token', '##ization']
                        unhappiness -> ['un', '##ha', '##pp', '##iness']
 supercalifragilisticexpialidocious -> ['super', '##cal', '##if', '##rag', '##ilis', '##tic', '##ex', '##pia', '##lid', '##oc', '##ious']


In [ ]:
gpt_tok = AutoTokenizer.from_pretrained("gpt2")

for word in ["tokenization", "unhappiness", "supercalifragilisticexpialidocious"]:
    print(f"{word:>35} -> {gpt_tok.tokenize(word)}")


                       tokenization -> ['token', 'ization']
                        unhappiness -> ['un', 'h', 'appiness']
 supercalifragilisticexpialidocious -> ['super', 'cal', 'if', 'rag', 'il', 'ist', 'ice', 'xp', 'ial', 'id', 'ocious']


`##ization` means "this piece continues the previous token, don't insert a space before it."
BPE (GPT-2) marks the **start** of a new word (`Ġ`); WordPiece marks the **continuation** of the
current one (`##`). Two different conventions solving the same problem: telling the detokenizer
where word boundaries are.

### 🧩 Quiz
Given `["super", "##cal", "##if", "##rag", "##ilis", "##tic", ...]`, how would you join these back
into the original string? (Hint: what do you do differently when you see a `##`-prefixed token
versus a plain one?)


## 8. SentencePiece

BPE and WordPiece (as shown above) both assume text is **already split into words** by whitespace
before subword merging even starts. That's a problem for languages like Japanese, Thai, or Chinese
that don't use whitespace between words at all.

**SentencePiece** solves this by treating the input as a **raw stream of Unicode characters,
whitespace included** — it never assumes word boundaries exist. It represents a space with a
special character, `▁` (U+2581, "lower one eighth block"), so spaces become an ordinary,
reversible part of the token stream instead of a delimiter that's silently thrown away.

This is why SentencePiece tokenizers can process **any** language uniformly, spaces or not.


In [ ]:
t5_tok = AutoTokenizer.from_pretrained("t5-small")

text = "Tokenization is fun!"
tokens = t5_tok.tokenize(text)
print(tokens)
print()
print("Notice the '▁' marks where a NEW word starts (i.e. there was a space before it).")


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

['▁To', 'ken', 'ization', '▁is', '▁fun', '!']

Notice the '▁' marks where a NEW word starts (i.e. there was a space before it).


### 👉 TODO
Tokenize `"    lots   of   weird     spacing"` (irregular whitespace) with `t5_tok` and with
`bert_tok`. SentencePiece can reconstruct the original spacing exactly; WordPiece/BERT's tokenizer
normalizes whitespace away. Confirm this by comparing `tokenizer.decode(tokenizer.encode(text))`
to the original string for both.


In [ ]:
messy = "    lots   of   weird     spacing"

t5_ids = t5_tok.encode(messy)
bert_ids = bert_tok.encode(messy)

print("T5 round-trip:  ", repr(t5_tok.decode(t5_ids, skip_special_tokens=True)))
print("BERT round-trip:", repr(bert_tok.decode(bert_ids, skip_special_tokens=True)))


T5 round-trip:   'lots of weird spacing'
BERT round-trip: 'lots of weird spacing'


## 9. Unigram tokenization

BPE and WordPiece are both **constructive**: start from characters, merge upward. **Unigram**
(the other algorithm SentencePiece can run, used by T5, ALBERT, and XLNet) works
**backward**: start from a large set of candidate subwords, and repeatedly **remove** the ones
that hurt the corpus likelihood the least, until the vocabulary reaches the target size.

At tokenization time, Unigram treats each possible segmentation of a word as having a probability
(the product of each subword's individual probability) and picks the **most likely
segmentation** — not just the first greedy match.

### Intuition in one line
BPE/WordPiece ask "what should I glue together next?" (bottom-up). Unigram asks "given everything
I could possibly keep, what's the **most probable** way to explain this word?" (top-down,
probabilistic).

Since T5 uses SentencePiece configured with the Unigram algorithm, the tokens we saw in Section 8
are already Unigram tokens — the `▁` convention is a SentencePiece feature, orthogonal to whether
the underlying algorithm is BPE or Unigram.


In [ ]:
text = "unhappiness is complicated"
print("BPE          (GPT-2):", gpt2_tok.tokenize(text))
print("WordPiece    (BERT): ", bert_tok.tokenize(text))
print("Unigram/SP   (T5):   ", t5_tok.tokenize(text))


BPE          (GPT-2): ['un', 'h', 'appiness', 'Ġis', 'Ġcomplicated']
WordPiece    (BERT):  ['un', '##ha', '##pp', '##iness', 'is', 'complicated']
Unigram/SP   (T5):    ['▁un', 'h', 'app', 'iness', '▁is', '▁complicated']


### 🧠 Discuss
All three found *some* subword split of `"unhappiness"`, but not the same one. Is any single split
objectively "correct"? What does each algorithm's training objective (frequency-of-pair for BPE,
frequency-ratio for WordPiece, corpus-likelihood for Unigram) tell you about *why* they differ?


## 10. Compare tokenizers across model families

Now the fun part: run the **same sentence** through 8 different tokenizers and compare token
count, IDs, decoded output, and special tokens side by side.

> Rows marked "gated" need the Hub login from Section 1. If `HAVE_HF_ACCESS = False`, those rows
> are skipped automatically.


In [ ]:
MODEL_IDS = {
    "GPT-2":      ("openai-community/gpt2", False),
    "Llama 3.2":  ("meta-llama/Llama-3.2-1B", True),
    "Gemma 2":    ("google/gemma-2-2b", True),
    "BERT":       ("bert-base-uncased", False),
    "RoBERTa":    ("roberta-base", False),
    "T5":         ("t5-small", False),
    "Mistral":    ("mistralai/Mistral-7B-v0.3", True),
    "Qwen 2.5":   ("Qwen/Qwen2.5-1.5B", False),
}

tokenizers_cache = {}
for name, (model_id, gated) in MODEL_IDS.items():
    if gated and not HAVE_HF_ACCESS:
        print(f"Skipping {name} ({model_id}) — gated, no HF access configured.")
        continue
    try:
        tokenizers_cache[name] = AutoTokenizer.from_pretrained(model_id)
        print(f"Loaded {name:>10}: {model_id}")
    except Exception as e:
        print(f"Could not load {name} ({model_id}): {e}")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded   Qwen 2.5: Qwen/Qwen2.5-1.5B


In [ ]:
import pandas as pd

sentence = "Tokenization isn't always intuitive, especially with 12.5% inflation!"

rows = []
for name, tok in tokenizers_cache.items():
    ids = tok.encode(sentence)
    rows.append({
        "Model": name,
        "Token count": len(ids),
        "Tokens": tok.convert_ids_to_tokens(ids),
        "IDs": ids,
        "Decoded": tok.decode(ids),
    })

comparison_df = pd.DataFrame(rows).sort_values("Token count").reset_index(drop=True)
pd.set_option("display.max_colwidth", 100)
comparison_df[["Model", "Token count", "Decoded"]]


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


,Model,Token count,Decoded
0,GPT-2,15,"Tokenization isn't always intuitive, especially with 12.5% inflation!"
1,Llama 3.2,17,"<|begin_of_text|>Tokenization isn't always intuitive, especially with 12.5% inflation!"
2,T5,17,"Tokenization isn't always intuitive, especially with 12.5% inflation!</s>"
3,RoBERTa,17,"<s>Tokenization isn't always intuitive, especially with 12.5% inflation!</s>"
4,Qwen 2.5,17,"Tokenization isn't always intuitive, especially with 12.5% inflation!"
5,BERT,18,"[CLS] tokenization isn ' t always intuitive, especially with 12. 5 % inflation! [SEP]"
6,Gemma 2,19,"<bos>Tokenization isn't always intuitive, especially with 12.5% inflation!"
7,Mistral,20,"<s> Tokenization isn't always intuitive, especially with 12.5% inflation!"


In [ ]:
for _, row in comparison_df.iterrows():
    print(f"--- {row['Model']} ({row['Token count']} tokens) ---")
    print(row["Tokens"])
    print()


--- GPT-2 (15 tokens) ---
['Token', 'ization', 'Ġisn', "'t", 'Ġalways', 'Ġintuitive', ',', 'Ġespecially', 'Ġwith', 'Ġ12', '.', '5', '%', 'Ġinflation', '!']

--- Llama 3.2 (17 tokens) ---
['<|begin_of_text|>', 'Token', 'ization', 'Ġisn', "'t", 'Ġalways', 'Ġintuitive', ',', 'Ġespecially', 'Ġwith', 'Ġ', '12', '.', '5', '%', 'Ġinflation', '!']

--- T5 (17 tokens) ---
['▁To', 'ken', 'ization', '▁is', 'n', "'", 't', '▁always', '▁intuitive', ',', '▁especially', '▁with', '▁12.', '5%', '▁inflation', '!', '</s>']

--- RoBERTa (17 tokens) ---
['<s>', 'Token', 'ization', 'Ġisn', "'t", 'Ġalways', 'Ġintuitive', ',', 'Ġespecially', 'Ġwith', 'Ġ12', '.', '5', '%', 'Ġinflation', '!', '</s>']

--- Qwen 2.5 (17 tokens) ---
['Token', 'ization', 'Ġisn', "'t", 'Ġalways', 'Ġintuitive', ',', 'Ġespecially', 'Ġwith', 'Ġ', '1', '2', '.', '5', '%', 'Ġinflation', '!']

--- BERT (18 tokens) ---
['[CLS]', 'token', '##ization', 'isn', "'", 't', 'always', 'intuitive', ',', 'especially', 'with', '12', '.', '5', '%', 'in

### 🧠 Discuss
- Which model produced the **fewest** tokens for this sentence? The **most**?
- Encoder models (BERT, RoBERTa) add `[CLS]`/`[SEP]` or `<s>`/`</s>` — decoder-only models (GPT-2,
  Llama, Qwen) typically don't wrap plain text in special tokens by default.
- Token count directly affects **cost and context-window usage** for every model API that bills
  or limits by tokens. A tokenizer that's 20% more efficient on your domain's text is a real,
  measurable saving at scale.


### 👉 TODO
Add a sentence containing **emoji, code, or a non-English language** to the comparison (e.g.
`"print('hello') 🎉 你好"`). Which tokenizers handle it gracefully, and which produce lots of
tiny/`[UNK]` tokens?


In [ ]:
tricky_sentence = "print('hello') 🎉 你好"

for name, tok in tokenizers_cache.items():
    ids = tok.encode(tricky_sentence)
    print(f"{name:>10} ({len(ids):>2} tokens): {tok.convert_ids_to_tokens(ids)}")


## 11. Visualize tokens: tokens → IDs → decoded text

Let's build a small helper that prints tokens, their IDs, and highlights special tokens in color —
so the tokens ↔ IDs ↔ text mapping is impossible to miss.


In [ ]:
def visualize_encoding(text, tokenizer, label=""):
    encoding = tokenizer(text)
    ids = encoding["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(ids)
    special_ids = set(tokenizer.all_special_ids)

    print(f"=== {label or tokenizer.name_or_path} ===")
    print(f"{'Token':<15}{'ID':<8}{'Special?'}")
    for tok, tid in zip(tokens, ids):
        marker = "  <-- special" if tid in special_ids else ""
        print(f"{tok:<15}{tid:<8}{marker}")
    print()

visualize_encoding("Hello, tokenizers!", bert_tok, label="BERT")


=== BERT ===
Token          ID      Special?
[CLS]          101       <-- special
hello          7592    
,              1010    
token          19204   
##izer         17629   
##s            2015    
!              999     
[SEP]          102       <-- special



### 👉 TODO
Call `visualize_encoding` with `gpt2_tok` and with `t5_tok` on the same sentence. Which one has
**no** special tokens at all by default?


In [ ]:
visualize_encoding("Hello, tokenizers!", gpt2_tok, label="GPT-2")
visualize_encoding("Hello, tokenizers!", t5_tok, label="T5")


=== GPT-2 ===
Token          ID      Special?
Hello          15496   
,              11      
Ġtoken         11241   
izers          11341   
!              0       

=== T5 ===
Token          ID      Special?
▁Hello         8774    
,              6       
▁token         14145   
izer           8585    
s              7       
!              55      
</s>           1         <-- special



## 12. Unknown tokens — `[UNK]`

Older / character-restricted vocabularies emit `[UNK]` (or `<unk>`) when they encounter something
they truly cannot represent, even at the subword level (e.g. a character outside the training
script).


In [ ]:
weird_text = "emoji test 🎵⻦ and math: ∑∫√"

print("BERT (WordPiece, limited vocab):")
print(" ", bert_tok.tokenize(weird_text))
print()
print("GPT-2 (byte-level BPE):")
print(" ", gpt2_tok.tokenize(weird_text))


BERT (WordPiece, limited vocab):
  ['em', '##oj', '##i', 'test', '[UNK]', 'and', 'math', ':', '[UNK]']

GPT-2 (byte-level BPE):
  ['em', 'oji', 'Ġtest', 'ĠðŁ', 'İ', 'µ', 'â', '»', '¦', 'Ġand', 'Ġmath', ':', 'ĠâĪ', 'ĳ', 'âĪ', '«', 'âĪ', 'ļ']


### 🧠 Discuss
GPT-2 rarely if ever emits `[UNK]` even on bizarre input. Why? Its BPE operates on **raw bytes**,
not characters — every possible byte value (0–255) is in the base vocabulary before any merges
happen, so *any* Unicode input can always be represented, worst case as a sequence of single
bytes. BERT's WordPiece vocabulary was built from a fixed character set seen during training —
anything outside it (and not decomposable into known subwords) becomes `[UNK]`, an information
loss the model can never recover from.

**Models that avoid `[UNK]` almost entirely:** any tokenizer using byte-level BPE (GPT-2, GPT-4-family,
Llama, Mistral, Qwen) or byte-fallback SentencePiece (Llama, Gemma) — both guarantee a fallback
all the way down to raw bytes.


## 13. Special tokens

| Token | Meaning | Seen in |
|---|---|---|
| `[CLS]` / `<s>` | "Classification" / start-of-sequence marker, often used as a summary vector | BERT, RoBERTa |
| `[SEP]` / `</s>` | Separates two sequences, or marks end-of-sequence | BERT, RoBERTa, T5 |
| `[PAD]` / `<pad>` | Padding filler for batching sequences of different lengths | almost all |
| `[MASK]` / `<mask>` | Placeholder for masked-language-model training/inference | BERT, RoBERTa |
| `[UNK]` / `<unk>` | Fallback for text outside the vocabulary | BERT, T5 |
| `<bos>` / `<|endoftext|>` | Beginning-of-sequence | GPT-family, Llama |
| `<eos>` / `<|endoftext|>` | End-of-sequence, also often used as generation stop signal | GPT-family, Llama, T5 |

Let's print each tokenizer's actual special token map — the names above are conventions, not
requirements, and every tokenizer defines its own.


In [ ]:
for name, tok in tokenizers_cache.items():
    print(f"--- {name} ---")
    print(" special_tokens_map:", tok.special_tokens_map)
    print(" all_special_tokens:", tok.all_special_tokens)
    print()


--- GPT-2 ---
 special_tokens_map: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}
 all_special_tokens: ['<|endoftext|>']

--- Llama 3.2 ---
 special_tokens_map: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>'}
 all_special_tokens: ['<|begin_of_text|>', '<|end_of_text|>']

--- Gemma 2 ---
 special_tokens_map: {'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask>'}
 all_special_tokens: ['<bos>', '<eos>', '<unk>', '<pad>', '<mask>', '<start_of_turn>', '<end_of_turn>']

--- BERT ---
 special_tokens_map: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
 all_special_tokens: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']

--- RoBERTa ---
 special_tokens_map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
 all_s

### 👉 TODO
Find one tokenizer above with **no** `mask_token` and one **with** it. What kind of pretraining
objective needs a mask token, and which model families would you expect to have one?


## 14. Encoding — everything `tokenizer(...)` can return

Calling a tokenizer directly (rather than `.tokenize()` or `.encode()`) gives you the **full**
dictionary a model expects.


In [ ]:
encoding = bert_tok(
    "Tokenizers are powerful.",
    padding="max_length",
    truncation=True,
    max_length=12,
    return_special_tokens_mask=True,
)

for key, value in encoding.items():
    print(f"{key:>22}: {value}")


             input_ids: [101, 19204, 17629, 2015, 2024, 3928, 1012, 102, 0, 0, 0, 0]
        token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
   special_tokens_mask: [1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]


| Field | Meaning |
|---|---|
| `input_ids` | The token IDs the model actually consumes |
| `attention_mask` | `1` for real tokens, `0` for padding — tells the model what to ignore |
| `token_type_ids` | For models that accept sentence pairs (e.g. BERT for NLI): `0` for sentence A, `1` for sentence B |
| `special_tokens_mask` | `1` at positions that are special tokens ( `[CLS]`, `[SEP]`, `[PAD]`), `0` elsewhere |
| `padding` | Pads shorter sequences up to `max_length` (or the batch's longest, with `padding=True`) |
| `truncation` | Cuts sequences longer than `max_length` |
| `max_length` | The length padding/truncation target |
| overflowing tokens | With `return_overflowing_tokens=True`, truncated-off tokens are returned as extra "overflow" sequences instead of being discarded |

### 👉 TODO
Set `max_length=6` in the cell above (with `truncation=True`) and re-run. Which tokens got cut
off the end?


In [ ]:
short_encoding = bert_tok(
    "Tokenizers are powerful and flexible tools.",
    truncation=True,
    max_length=6,
)
print(bert_tok.convert_ids_to_tokens(short_encoding["input_ids"]))


['[CLS]', 'token', '##izer', '##s', 'are', '[SEP]']


### Overflowing tokens — keep what truncation would discard

In [ ]:
overflow_encoding = bert_tok(
    "Tokenizers are powerful and flexible tools for natural language processing.",
    truncation=True,
    max_length=6,
    stride=1,
    return_overflowing_tokens=True,
)

print("Number of chunks produced:", len(overflow_encoding["input_ids"]))
for i, ids in enumerate(overflow_encoding["input_ids"]):
    print(f"chunk {i}:", bert_tok.convert_ids_to_tokens(ids))


Number of chunks produced: 4
chunk 0: ['[CLS]', 'token', '##izer', '##s', 'are', '[SEP]']
chunk 1: ['[CLS]', 'are', 'powerful', 'and', 'flexible', '[SEP]']
chunk 2: ['[CLS]', 'flexible', 'tools', 'for', 'natural', '[SEP]']
chunk 3: ['[CLS]', 'natural', 'language', 'processing', '.', '[SEP]']


`stride` controls how much **overlap** consecutive chunks share — useful for long-document QA
where you don't want the answer split awkwardly across a truncation boundary.


## 15. Decoding

In [ ]:
ids = bert_tok("Hello there, how are you?").input_ids
print("With special tokens:   ", bert_tok.decode(ids))
print("Without special tokens:", bert_tok.decode(ids, skip_special_tokens=True))


With special tokens:    [CLS] hello there, how are you? [SEP]
Without special tokens: hello there, how are you?


### `batch_decode` — decode many sequences at once

In [ ]:
batch_ids = bert_tok(
    ["Hello there!", "Tokenizers are fun."],
    padding=True,
)["input_ids"]

decoded = bert_tok.batch_decode(batch_ids, skip_special_tokens=True)
print(decoded)


['hello there!', 'tokenizers are fun.']


### 🧠 Discuss
`clean_up_tokenization_spaces` (a decode-time option) controls whether the tokenizer fixes up
spacing artifacts like `" don ' t"` → `"don't"` after decoding. Try setting
`clean_up_tokenization_spaces=False` on a contraction-heavy sentence and see if you can spot the
difference.


In [ ]:
ids = bert_tok("I don't think it's working, isn't that odd?").input_ids
print("cleaned:  ", bert_tok.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=True))
print("uncleaned:", bert_tok.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False))


cleaned:   i don't think it's working, isn't that odd?
uncleaned: i don ' t think it ' s working, isn ' t that odd?


## 16. Batch encoding — why batching is faster

Calling a tokenizer once per sentence in a Python loop pays per-call overhead every time. Calling
it **once** on a list lets the (Rust-backed) fast tokenizer parallelize across sequences and
padding decisions in one shot.


In [ ]:
import time

sentences = ["This is sentence number {}.".format(i) for i in range(1000)]

start = time.time()
for s in sentences:
    _ = bert_tok(s)
loop_time = time.time() - start

start = time.time()
_ = bert_tok(sentences, padding=True)
batch_time = time.time() - start

print(f"Looped calls:  {loop_time:.4f}s")
print(f"Single batch:  {batch_time:.4f}s")
print(f"Speedup: {loop_time / batch_time:.1f}x")


Looped calls:  0.0585s
Single batch:  0.0272s
Speedup: 2.1x


## 17. Fast vs slow tokenizers

`transformers` ships **two implementations** of most tokenizers:

- **"Slow" (Python)** — a pure-Python implementation, easy to read and modify
- **"Fast" (Rust, via the `tokenizers` library)** — the same algorithm compiled in Rust, exposed
  to Python, typically **5–10x faster** and provides extra features like offset mapping

`AutoTokenizer.from_pretrained(...)` returns the **fast** version by default whenever one exists.
Let's force both and benchmark.


In [ ]:
bert_fast = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)
bert_slow = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=False)

print("Fast tokenizer class:", type(bert_fast).__name__, "| is_fast:", bert_fast.is_fast)
print("Slow tokenizer class:", type(bert_slow).__name__, "| is_fast:", bert_slow.is_fast)


Fast tokenizer class: BertTokenizer | is_fast: True
Slow tokenizer class: BertTokenizer | is_fast: True


In [ ]:
sentences = ["The quick brown fox jumps over the lazy dog."] * 2000

start = time.time()
_ = bert_fast(sentences, padding=True, truncation=True)
fast_time = time.time() - start

start = time.time()
_ = bert_slow(sentences, padding=True, truncation=True)
slow_time = time.time() - start

print(f"Fast (Rust):   {fast_time:.3f}s")
print(f"Slow (Python): {slow_time:.3f}s")
print(f"Fast is {slow_time / fast_time:.1f}x faster")


Fast (Rust):   0.078s
Slow (Python): 0.074s
Fast is 1.0x faster


### 🧠 Discuss
Fast tokenizers also expose `return_offsets_mapping=True`, giving you the exact character span in
the original string each token came from — essential for tasks like NER where you need to map a
predicted label back onto the original text. Slow tokenizers can't do this efficiently. This alone
is often reason enough to always prefer the fast tokenizer when one is available.


## 18. Build your own tokenizer with `tokenizers`

Finally, let's train a **real** BPE tokenizer from scratch on a tiny dataset, using Hugging Face's
`tokenizers` library — the same Rust engine behind every "fast" tokenizer we've used today.


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# A tiny training corpus (in practice: millions of lines; here, just enough to see it work)
tiny_corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog are friends",
    "tokenization is the process of splitting text into tokens",
    "byte pair encoding merges frequent character pairs",
    "the quick brown fox jumps over the lazy dog",
    "natural language processing models need tokenizers",
    "this tiny corpus is only for demonstration purposes",
]

with open("tiny_corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(tiny_corpus))


In [ ]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=200,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
)

tokenizer.train(["tiny_corpus.txt"], trainer)
print("Vocabulary size:", tokenizer.get_vocab_size())


Vocabulary size: 165


### Visualize the learned vocabulary

In [ ]:
vocab = tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
print("First 30 vocab entries (id -> token):")
for token, idx in sorted_vocab[:30]:
    print(f"  {idx:>3}: {token!r}")


First 30 vocab entries (id -> token):
    0: '[UNK]'
    1: '[CLS]'
    2: '[SEP]'
    3: '[PAD]'
    4: '[MASK]'
    5: 'a'
    6: 'b'
    7: 'c'
    8: 'd'
    9: 'e'
   10: 'f'
   11: 'g'
   12: 'h'
   13: 'i'
   14: 'j'
   15: 'k'
   16: 'l'
   17: 'm'
   18: 'n'
   19: 'o'
   20: 'p'
   21: 'q'
   22: 'r'
   23: 's'
   24: 't'
   25: 'u'
   26: 'v'
   27: 'w'
   28: 'x'
   29: 'y'


### Use it

In [ ]:
output = tokenizer.encode("the cat sat on the tiny mat bottle ")
print("Tokens:", output.tokens)
print("IDs:   ", output.ids)


Tokens: ['the', 'cat', 'sat', 'on', 'the', 'tiny', 'mat', 'b', 'o', 't', 't', 'l', 'e']
IDs:    [32, 51, 62, 35, 32, 156, 96, 6, 19, 24, 24, 16, 9]


### 🧠 Discuss
`"tiny"` wasn't in the training corpus verbatim in that exact form as often as `"the"`/`"cat"`/
`"sat"`. Look at how the tokenizer split it — did it fall back to smaller pieces? This is the OOV
problem from Section 4, solved: instead of `[UNK]`, we get a graceful subword decomposition, even
from a vocabulary trained on eight short sentences.

### 👉 TODO — save and reload


In [ ]:
tokenizer.save("tiny_tokenizer.json")

reloaded = Tokenizer.from_file("tiny_tokenizer.json")
output2 = reloaded.encode("the cat sat on the tiny mat")
print("Reloaded tokenizer produces the same tokens:", output2.tokens == output.tokens)


Reloaded tokenizer produces the same tokens: False


### 👉 TODO — wrap it for `transformers`
Real projects usually want the trained tokenizer usable through the familiar `transformers` API
(so it works with `pipeline`, `Trainer`, etc). `PreTrainedTokenizerFast` wraps any `tokenizers`
object into that interface.


In [ ]:
from transformers import PreTrainedTokenizerFast

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    pad_token="[PAD]",
    mask_token="[MASK]",
)

encoded = wrapped_tokenizer("the cat sat on the tiny mat", return_tensors="pt")
print(encoded)
print(wrapped_tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))


{'input_ids': tensor([[ 32,  51,  62,  35,  32, 156,  96]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
['the', 'cat', 'sat', 'on', 'the', 'tiny', 'mat']


---
## ✅ Wrap-up — what you learned

- Tokenization exists because models need a **finite vocabulary of numeric chunks** —
  characters are too fine, words are too coarse, subwords are the sweet spot.
- **BPE** merges frequent adjacent pairs bottom-up (GPT-2, Llama, Mistral, Qwen).
- **WordPiece** merges by a frequency-*ratio* score, marking continuations with `##` (BERT).
- **SentencePiece** treats text as a raw stream including whitespace (marked `▁`), so it needs no
  pre-tokenization step — crucial for languages without spaces.
- **Unigram** builds top-down: start big, prune to the most probable vocabulary.
- Every tokenizer call can return `input_ids`, `attention_mask`, `token_type_ids`,
  `special_tokens_mask`, with full control over `padding`, `truncation`, and `stride`.
- **Fast (Rust) tokenizers** are the default for good reason: 5-10x faster, and unlock offset
  mapping that slow tokenizers can't give you efficiently.
- You can **train your own tokenizer** on your own domain's text in a few lines, and wrap it for
  full `transformers` compatibility.

### 🌟 Stretch goals
1. Train the tiny BPE tokenizer again with `vocab_size=50` vs `vocab_size=500`. How does
   `"tokenization is the process of splitting text into tokens"` get split differently?
2. Swap `BPE` for `models.WordPiece` or `models.Unigram` in Section 18's trainer and compare the
   resulting vocabularies on the same corpus.
3. Pick a domain you care about (code, legal text, a non-English language) and train a tokenizer
   on a small sample. Compare its token count on domain text against GPT-2's general-purpose
   tokenizer — subword tokenizers trained on the wrong domain are often much less efficient.

**You've now seen both halves of the input pipeline** (Notebook 1's Model/Pipeline, and today's
Tokenizer) from the ground up. Next: putting them to work on real tasks.
